# Project 3: Y-Splitter & MMI (1x2, 2x2) — Meep (open-source FDTD, 2D-effective-index + full 3D)
**ساختار:** ۰) نصب و اتصال به Drive ← ۱) ریاضیات self-imaging ← ۲) هندسه ← ۳) شبیه‌ساز ← ۴) اعتبارسنجی با مرجع (Tidy3D/PhotonForge: MMI 2x2 با W=4µm, L≈56.5µm, taper 1.5µm؛ Y-splitter مقاله Zhang et al., Opt. Express 21, 1310 (2013)) ← ۵) سوییپ‌ها ← ۶) حساسیت به خطای ساخت ← ۷) جدول نهایی.

**قاعدهٔ کش:** هر شبیه‌سازی یک بار اجرا و در Drive ذخیره می‌شود (`cache/*.json`). اگر Runtime ببندید و دوباره اجرا کنید، نتایج از Drive خوانده می‌شود و شبیه‌سازی تکرار نمی‌شود.

## ۰) نصب Meep (فقط بار اول). بعد از `condacolab.install()` runtime خودش restart می‌شود؛ سلول را دوباره اجرا کنید.

In [ ]:
# Meep روی pip نیست؛ از conda-forge نصب می‌شود.
try:
    import condacolab; condacolab.check()
except Exception:
    !pip -q install condacolab
    import condacolab; condacolab.install()   # runtime restart می‌شود

In [ ]:
!mamba install -y -q -c conda-forge pymeep=*=mpi_mpich_* nomkl matplotlib pandas > /dev/null 2>&1
import meep as mp; print('meep', mp.__version__)

## ۱) تنظیمات، Drive و سیستم کش

In [ ]:
import os, json, hashlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import meep as mp
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MMI_Project'
for d in ['cache','figs','tables']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# ---- ثابت‌های فیزیکی (واحد طول: µm) ----
N_SI, N_OX = 3.476, 1.444          # سیلیکون و SiO2 در 1550nm
T_SI   = 0.22                      # ضخامت لایه
N_SLAB = 2.85                      # n_eff مود TE0 اسلب 220nm (برای مدل 2D مؤثر؛ در 3D لازم نیست)
LAM0, LAM_MIN, LAM_MAX = 1.55, 1.45, 1.65
NF = 41                            # تعداد فرکانس‌ها
W0 = 0.5                           # عرض موجبر ورودی/خروجی
DPML = 1.0

def cached(key_dict):
    """دکوراتور ساده: نتیجه را بر اساس hash پارامترها در Drive ذخیره می‌کند."""
    h = hashlib.md5(json.dumps(key_dict, sort_keys=True).encode()).hexdigest()[:12]
    return f'{ROOT}/cache/{h}.json'

## ۲) ریاضیات (Soldano & Pennings, J. Lightwave Technol. 13, 615 (1995))
مود‌های MMI با عرض مؤثر $W_e$ (Goos–Hänchen):
$$W_e\simeq W+\frac{\lambda_0}{\pi}\Big(\frac{n_c}{n_r}\Big)^{2\sigma}(n_r^2-n_c^2)^{-1/2},\quad \sigma=0\ (TE)$$
طول beat دو مود پایین:
$$L_\pi=\frac{\pi}{\beta_0-\beta_1}\simeq\frac{4n_rW_e^2}{3\lambda_0}$$
فاصلهٔ N-تصویری (p=1):
- **تداخل عمومی (2x2):** $L=\dfrac{3L_\pi}{N}\Rightarrow L_{2\times2}^{3dB}=\dfrac{3L_\pi}{2}$ ، ورودی‌ها در $y=\pm W_e/6$ و خروجی‌ها هم همان‌جا.
- **تداخل متقارن (1x2، ورودی مرکزی):** $L=\dfrac{3L_\pi}{4N}\Rightarrow L_{1\times2}=\dfrac{3L_\pi}{8}$ ، خروجی‌ها در $y=\pm W_e/4$.

**معیارها** (با $T_i=|a_i|^2/|a_{ref}|^2$ ضریب مود‌ی توان خروجی i نسبت به موجبر مرجع):
$$IL=-10\log_{10}\sum_i T_i,\quad \text{Imbalance}=10\log_{10}\frac{T_1}{T_2},\quad \text{Split ratio}=\frac{T_1}{T_1+T_2}$$
$$\text{Excess loss}=IL-10\log_{10}N\ (\text{ایده‌آل}: 3.01\text{ dB برای 2 خروجی})$$
**پهنای باند ۱dB:** بازهٔ پیوستهٔ اطراف $\lambda_0$ که در آن $IL(\lambda)\le IL_{min}+1$ dB و هم‌زمان $|{\rm Imbalance}|\le 1$ dB. (هر دو را جدا گزارش می‌کنیم.)

In [ ]:
def self_imaging(W, n_r=N_SLAB, n_c=N_OX, lam=LAM0):
    We  = W + lam/np.pi*(n_r**2-n_c**2)**-0.5          # TE: sigma=0
    Lpi = 4*n_r*We**2/(3*lam)
    return dict(We=We, Lpi=Lpi, L_2x2=1.5*Lpi, L_1x2=3*Lpi/8,
                y_in_2x2=We/6, y_out_1x2=We/4)
for W in (4.0, 6.0): print(W, {k: round(v,2) for k,v in self_imaging(W).items()})
# تخمین اولیه است؛ طول واقعی با سوییپ FDTD پیدا می‌شود (مرجع Tidy3D: W=4 → L≈56.5µm در 3D).

## ۳) هندسه
`P` یک دیکشنری پارامتر است. `dw`=خطای عرض (همهٔ ویژگی‌های Si به‌اندازهٔ dw پهن می‌شوند، مدل over/under-etch)، `dL`=خطای طول MMI.
- `kind='2x2'`: ورودی‌ها و خروجی‌ها در $y=\pm sep/2$
- `kind='1x2'`: ورودی مرکزی، خروجی‌ها در $y=\pm sep/2$
- `kind='Y'`: Y-junction مقاله Zhang 2013 (۱۳ نقطهٔ عرض، طول ۲µm، خروجی‌ها با S-bend جدا می‌شوند، همان‌طور که در نوت‌بوک نمونه).

In [ ]:
Si = mp.Medium(index=N_SI)
Si2D = mp.Medium(index=N_SLAB)

def prism(pts, dim):
    mat = Si if dim==3 else Si2D
    h = T_SI if dim==3 else mp.inf
    z0 = -T_SI/2 if dim==3 else 0
    return mp.Prism([mp.Vector3(x,y,z0) for x,y in pts], height=h, axis=mp.Vector3(0,0,1), material=mat)

def block(x0,x1,yc,w,dim):
    return prism([(x0,yc-w/2),(x1,yc-w/2),(x1,yc+w/2),(x0,yc+w/2)], dim)

# مقاله Zhang 2013 (از نوت‌بوک نمونه)
Y_W = np.array((0.5,0.5,0.6,0.7,0.9,1.26,1.4,1.4,1.4,1.4,1.31,1.2,1.2))
Y_LEN, Y_YOUT = 2.0, 0.35

def build(P):
    """برمی‌گرداند: geometry, sx, x_src, x_mon, y_in, y_outs"""
    dim, kind = P['dim'], P['kind']; dw = P.get('dw',0.0)
    geo=[]
    if kind in ('2x2','1x2'):
        W, L, wt, Lt, sep = P['W']+dw, P['L']+P.get('dL',0.0), P['wt']+dw, P['Lt'], P['sep']
        w0 = W0+dw
        yin  = [-sep/2, sep/2] if kind=='2x2' else [0.0]
        yout = [-sep/2, sep/2]
        xe = L/2 + Lt                          # انتهای تیپر
        sx = 2*xe + 2*(2+DPML)
        geo.append(block(-L/2, L/2, 0, W, dim))                       # ناحیه MMI
        for y in yin:                                                 # ورودی: موجبر + تیپر
            geo.append(block(-sx/2-1, -xe, y, w0, dim))
            geo.append(prism([(-xe,y-w0/2),(-L/2,y-wt/2),(-L/2,y+wt/2),(-xe,y+w0/2)], dim))
        for y in yout:                                                # خروجی
            geo.append(block(xe, sx/2+1, y, w0, dim))
            geo.append(prism([(xe,y-w0/2),(L/2,y-wt/2),(L/2,y+wt/2),(xe,y+w0/2)], dim))
        sy = W + 2*2 + 2*DPML
        xs, xm = -xe-1.0, xe+1.0
    else:  # Y-junction + S-bend کسینوسی
        w = Y_W + dw; xs_ = np.linspace(0, Y_LEN, len(w))
        up = [(x, +ww/2) for x,ww in zip(xs_,w)]; lo=[(x,-ww/2) for x,ww in zip(xs_,w)]
        so, sl = P['s_off'], P['s_len']                              # آفست و طول S-bend
        wo = W0+dw
        xx = np.linspace(0, sl, 40); yy = so*(1-np.cos(np.pi*xx/sl))/2
        xb = Y_LEN
        # هر S-bend به صورت چندضلعی (لبه بالا/پایین ± wo/2)
        for sgn in (+1,-1):
            c = sgn*(Y_YOUT + yy)
            top = [(xb+x, y+wo/2) for x,y in zip(xx,c)]
            bot = [(xb+x, y-wo/2) for x,y in zip(xx,c)][::-1]
            geo.append(prism(top+bot, dim))
        geo.append(prism(up+lo[::-1], dim))
        xe0 = Y_LEN + sl
        yout = [-(Y_YOUT+so), Y_YOUT+so]; yin=[0.0]
        sx = xe0 + 2*(2+DPML) + 2     # ورودی از x=-?؛ انتقال دهیم تا مرکز سلول در صفر باشد
        shift = -(xe0)/2
        # جابه‌جایی همهٔ منشورها در x (ساده‌تر: cell center را جابه‌جا می‌کنیم)
        geo.append(block(-(2+DPML)-1+0.0, 0.0, 0.0, W0+dw, dim))      # موجبر ورودی
        P['_cx'] = xe0/2
        xs, xm = -1.0, xe0+1.0
        sy = (Y_YOUT+so)*2 + 2*2 + 2*DPML
    sz = (T_SI + 2*1.5 + 2*DPML) if dim==3 else 0
    cell = mp.Vector3(sx if kind!='Y' else sx, sy, sz)
    return geo, cell, xs, xm, yin, yout, P.get('_cx',0.0)

## ۴) شبیه‌ساز
- منبع: `EigenModeSource` مود TE0 (در Meep: `EVEN_Z`) با طیف گاوسی پهن (۱.۴۵–۱.۶۵µm) ← یک اجرا برای کل باند.
- مانیتور: `get_eigenmode_coefficients` روی مود پایه هر خروجی (فقط توان *در مود TE0* حساب می‌شود، پس تلفات تابشی و مود‌های بالاتر لحاظ می‌شوند).
- نرمال‌سازی: اجرای یک موجبر مستقیم (کش می‌شود).
- `dim=2`: مدل n_eff (سریع، برای سوییپ). `dim=3`: **۳بعدی کامل** (برای طرح نهایی و اعتبارسنجی).

In [ ]:
FC = 0.5*(1/LAM_MIN+1/LAM_MAX); DF = 1/LAM_MIN-1/LAM_MAX
FREQS = np.linspace(1/LAM_MAX, 1/LAM_MIN, NF)           # صعودی در f ⇒ λ نزولی
LAMS  = 1/FREQS

def _sim(geo, cell, xs, xm, yin, yout, dim, res, cx, ref=False):
    sz = cell.z
    dz_s = (T_SI+2*1.0) if dim==3 else 0
    wy = min(2.0, abs(yout[1]-yout[0])-0.5) if len(yout)>1 else 2.0
    wy = max(wy, 1.2)
    src = [mp.EigenModeSource(mp.GaussianSource(FC, fwidth=DF), center=mp.Vector3(xs-cx*0, yin[0]),
            size=mp.Vector3(0, wy, dz_s), eig_band=1, eig_parity=mp.EVEN_Z, eig_match_freq=True)]
    sim = mp.Simulation(cell_size=cell, geometry=geo, sources=src, resolution=res,
        boundary_layers=[mp.PML(DPML)], default_material=mp.Medium(index=N_OX if dim==3 else 1.0*N_OX),
        dimensions=dim, force_complex_fields=False, geometry_center=mp.Vector3(cx,0,0))
    mons = [sim.add_mode_monitor(FREQS, mp.ModeRegion(center=mp.Vector3(xm,y), size=mp.Vector3(0,wy,dz_s)))
            for y in (yin if ref else yout)]
    sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ey, mp.Vector3(xm, (yin if ref else yout)[0]), 1e-6))
    a = [np.abs(sim.get_eigenmode_coefficients(m,[1],eig_parity=mp.EVEN_Z).alpha[0,:,0])**2 for m in mons]
    ph = [np.angle(sim.get_eigenmode_coefficients(m,[1],eig_parity=mp.EVEN_Z).alpha[0,:,0]) for m in mons]
    return np.array(a), np.array(ph)

In [ ]:
def reference(dim, res):
    P = dict(dim=dim, kind='2x2', W=4, L=1, wt=0.5, Lt=0.5, sep=2.0)   # موجبر مستقیم (MMI کوتاه با wt=w0 ≈ مستقیم نیست)
    f = cached(dict(ref=1, dim=dim, res=res))
    if os.path.exists(f): return np.array(json.load(open(f)))
    # موجبر مستقیم واقعی
    sx, sy = 12.0, 4.0+2*DPML; sz = (T_SI+2*1.5+2*DPML) if dim==3 else 0
    geo=[block(-sx/2-1, sx/2+1, 0.0, W0, dim)]
    a,_ = _sim(geo, mp.Vector3(sx,sy,sz), -3.0, 3.0, [0.0], [0.0], dim, res, 0.0, ref=True)
    json.dump(a[0].tolist(), open(f,'w')); return a[0]

def simulate(P, res):
    """اجرای (یا خواندن از کش) یک شبیه‌سازی. خروجی: T (nout×NF) و فاز."""
    key = dict(P={k:v for k,v in P.items() if not k.startswith('_')}, res=res, lam=[LAM_MIN,LAM_MAX,NF])
    f = cached(key)
    if os.path.exists(f):
        d=json.load(open(f)); return np.array(d['T']), np.array(d['ph'])
    t0=time.time(); ref = reference(P['dim'], res)
    geo, cell, xs, xm, yin, yout, cx = build(dict(P))
    a, ph = _sim(geo, cell, xs, xm, yin, yout, P['dim'], res, cx)
    T = a/ref[None,:]
    json.dump(dict(T=T.tolist(), ph=ph.tolist(), time=time.time()-t0), open(f,'w'))
    print(f'  done in {time.time()-t0:.0f}s  | {key["P"]}')
    return T, ph

> ⚠️ **نکتهٔ مهم برای تیم/شما:** من (نویسندهٔ کد) Meep را در محیط خودم نداشتم و کد را اجرا نکردم. پس خروجی‌ها را با **تست‌های اعتبارسنجی بخش ۵** (موجبر مستقیم ⇒ T≈1، تقارن، پایستگی توان، و مقایسه با اعداد مرجع) بررسی کنید. اگر خطایی در API دیدید متن خطا را بفرستید تا اصلاح کنم.

## ۵) توابع معیار (IL، split ratio، imbalance، پهنای باند ۱dB)

In [ ]:
def metrics(T, kind='2x2'):
    """T: (nout, NF). برای 2x2 با ورود از P0: T[0]=bar, T[1]=cross (ترتیب y منفی→مثبت؛ ورودی در y منفی ⇒ T[0]=bar)."""
    tot = T.sum(0); IL = -10*np.log10(tot)
    imb = 10*np.log10(T[0]/T[1]); ratio = T[0]/tot
    i0 = np.argmin(abs(LAMS-LAM0))
    def band(mask):                       # عرض بازهٔ پیوستهٔ شامل مرکز
        if not mask[i0]: return 0.0
        lo=i0; hi=i0
        while lo>0 and mask[lo-1]: lo-=1
        while hi<len(mask)-1 and mask[hi+1]: hi+=1
        return abs(LAMS[lo]-LAMS[hi])*1e3         # nm
    bw_il  = band(IL <= IL.min()+1.0)
    bw_imb = band(abs(imb) <= 1.0)
    bw_both= band((IL <= IL.min()+1.0) & (abs(imb)<=1.0))
    return dict(IL_dB=IL[i0], EL_dB=IL[i0]-10*np.log10(2), imbalance_dB=imb[i0],
                split_ratio=ratio[i0], T0=T[0][i0], T1=T[1][i0],
                BW1dB_IL_nm=bw_il, BW1dB_imb_nm=bw_imb, BW1dB_both_nm=bw_both)

def save_fig(name): plt.savefig(f'{ROOT}/figs/{name}.png', dpi=200, bbox_inches='tight')

## ۶) اعتبارسنجی (حتماً اول انجام دهید)
1. **تست موجبر مستقیم:** باید T≈1 (خطای <۲٪).
2. **همگرایی رزولوشن** MMI.
3. **مقایسه با مرجع:** MMI 2x2 با W=4µm، wt=1.5µm، Lt=5µm، sep=2µm در 3D باید حول L≈56.5µm نقطهٔ ۳dB (bar≈cross≈0.5) و تلفات کم (مجموع ≳۰.۹۵ طبق نوت‌بوک مرجع) بدهد. Y-splitter مقاله: تلفات اضافه ≈۰.۲–۰.۳ dB (اندازه‌گیری مقاله 0.28±0.02 dB) و split≈50/50.

In [ ]:
# تست 1: موجبر مستقیم با خودش نرمال می‌شود => 1. (یک بار)
a = reference(2, 20); print('ref mean power coeff', a.mean())
# تست 2: همگرایی رزولوشن، 2D، L=56.5
for r in (16, 24, 32):
    T,_ = simulate(dict(dim=2,kind='2x2',W=4.0,L=56.5,wt=1.5,Lt=5.0,sep=2.0), r)
    m = metrics(T); print(r, {k:round(float(v),3) for k,v in m.items() if k in ('T0','T1','IL_dB')})

## ۷) مرحله ۱ و ۲: سوییپ طول — 2x2 و 1x2 (مدل 2D مؤثر، سریع)

In [ ]:
RES2D = 24
def sweep(name, base, key, values, res=RES2D):
    rows=[]
    for v in values:
        P=dict(base); P[key]=float(v); T,_=simulate(P,res)
        m=metrics(T, base['kind']); m[key]=float(v); rows.append(m)
    df=pd.DataFrame(rows); df.to_csv(f'{ROOT}/tables/{name}.csv', index=False); return df

base22 = dict(dim=2,kind='2x2',W=4.0,L=56.5,wt=1.5,Lt=5.0,sep=2.0)
df_L22 = sweep('sweepL_2x2', base22, 'L', np.arange(20,81,2.0))
base12 = dict(dim=2,kind='1x2',W=6.0,L=20.0,wt=1.5,Lt=5.0,sep=3.0)   # ±W_e/4 ≈ ±1.6
df_L12 = sweep('sweepL_1x2', base12, 'L', np.arange(8,41,1.0))

fig,ax=plt.subplots(1,2,figsize=(11,4))
for a_,df,t in zip(ax,(df_L22,df_L12),('2x2 MMI (W=4µm)','1x2 MMI (W=6µm)')):
    a_.plot(df.L,df.T0,label='T0 (bar / out−)'); a_.plot(df.L,df.T1,label='T1 (cross / out+)')
    a_.plot(df.L,df.T0+df.T1,'k--',label='total'); a_.set_xlabel('L (µm)'); a_.set_title(t); a_.legend(); a_.grid(alpha=.3)
save_fig('sweep_length'); plt.show()

In [ ]:
def best_L(df, Lmin):
    d=df[(df.L>Lmin)&((df.T0+df.T1)>0.85)]
    return float(d.loc[(d.T0-d.T1).abs().idxmin(),'L'])
L22 = best_L(df_L22, 45); L12 = best_L(df_L12, 10)
print('طول بهینه 2x2:', L22, ' | 1x2:', L12, ' | تخمین تئوری:', self_imaging(4)['L_2x2'], self_imaging(6)['L_1x2'])

## ۸) مرحله ۳: سوییپ عرض و هندسهٔ تیپر (ورودی/خروجی)

In [ ]:
best22 = dict(base22, L=L22)
df_W   = sweep('sweepW_2x2',  best22, 'W',  [3.5,4.0,4.5,5.0])       # (L ثابت؛ برای دقت بیشتر L را برای هر W از تئوری L∝W_e² مقیاس کنید)
df_wt  = sweep('sweepwt_2x2', best22, 'wt', [0.8,1.0,1.2,1.5,1.8,2.0])
df_Lt  = sweep('sweepLt_2x2', best22, 'Lt', [1,2,3,5,8,12])

# عرض با مقیاس‌گذاری هوشمند L ∝ W_e^2 (منصفانه‌تر)
rows=[]
for W in [3.5,4.0,4.5,5.0]:
    Ls = self_imaging(W)['L_2x2']*L22/self_imaging(4.0)['L_2x2']
    T,_=simulate(dict(best22,W=W,L=round(Ls,2)),RES2D); m=metrics(T); m.update(W=W,L=round(Ls,2)); rows.append(m)
df_Wscaled=pd.DataFrame(rows); df_Wscaled.to_csv(f'{ROOT}/tables/sweepW_scaledL_2x2.csv',index=False)

fig,ax=plt.subplots(1,3,figsize=(14,3.8))
for a_,df,k in zip(ax,(df_Wscaled,df_wt,df_Lt),('W','wt','Lt')):
    a_.plot(df[k],df.IL_dB,'o-',label='IL (dB)'); a_.plot(df[k],df.imbalance_dB.abs(),'s-',label='|imbalance| (dB)')
    a_.set_xlabel(k+' (µm)'); a_.grid(alpha=.3); a_.legend()
save_fig('sweep_geometry'); plt.show()

## ۹) Y-splitter (Zhang 2013): سوییپ طول S-bend و طیف

In [ ]:
baseY = dict(dim=2,kind='Y',s_off=1.0,s_len=5.0)
rowsY=[]
for sl in (3,5,8,12):
    T,_=simulate(dict(baseY,s_len=float(sl)),RES2D); m=metrics(T); m['s_len']=sl; rowsY.append(m)
df_Y=pd.DataFrame(rowsY); df_Y.to_csv(f'{ROOT}/tables/Y_sbend.csv',index=False); df_Y[['s_len','IL_dB','EL_dB','imbalance_dB','split_ratio']]

In [ ]:
# طیف Y (مرجع: IL ≈ 3.01+~0.1..0.3 dB)
T,_=simulate(dict(baseY,s_len=8.0),RES2D)
fig,ax=plt.subplots(figsize=(6,4)); ax.plot(LAMS,-10*np.log10(T.sum(0)),label='IL total'); ax.plot(LAMS,-10*np.log10(T[0]),label='port −'); ax.plot(LAMS,-10*np.log10(T[1]),label='port +')
ax.set_xlabel('λ (µm)'); ax.set_ylabel('dB'); ax.legend(); ax.grid(alpha=.3); save_fig('Y_spectrum'); plt.show()

## ۱۰) مرحله ۴: حساسیت به خطای ساخت (عرض ±۲۰nm، طول ±۱µm)

In [ ]:
def fab(name, base, key, deltas, res=RES2D):
    rows=[]
    for d in deltas:
        T,_=simulate(dict(base,**{key:float(d)}),res); m=metrics(T); m[key]=d; rows.append(m)
    df=pd.DataFrame(rows); df.to_csv(f'{ROOT}/tables/{name}.csv',index=False); return df
bestcfg = dict(best22)
df_dw = fab('fab_dw_2x2', bestcfg, 'dw', [-0.02,-0.01,0,0.01,0.02])
df_dL = fab('fab_dL_2x2', bestcfg, 'dL', [-1,-0.5,0,0.5,1])
bY = dict(baseY, s_len=8.0)
df_dwY = fab('fab_dw_Y', bY, 'dw', [-0.02,-0.01,0,0.01,0.02])
fig,ax=plt.subplots(1,3,figsize=(14,3.8))
for a_,df,k in zip(ax,(df_dw,df_dL,df_dwY),('dw','dL','dw')):
    a_.plot(df[k]*1e3 if k=='dw' else df[k], df.IL_dB,'o-',label='IL'); a_.plot(df[k]*1e3 if k=='dw' else df[k], df.imbalance_dB.abs(),'s-',label='|Imb|')
    a_.set_xlabel(k+(' (nm)' if k=='dw' else ' (µm)')); a_.grid(alpha=.3); a_.legend()
save_fig('fab_sensitivity'); plt.show()

## ۱۱) تأیید ۳بعدی کامل (طرح نهایی) — **سنگین‌ترین بخش**
هر اجرای 3D با res=16..20 روی Colab معمولاً ده‌ها دقیقه است (CPU). برای سبک‌تر شدن: Colab Pro (RAM بالا) یا کاهش NF. نتیجه در Drive کش می‌شود. اول res=14 را امتحان کنید، بعد 20.

In [ ]:
RES3D = 16
P3 = dict(base22, dim=3, L=56.5)          # نقطهٔ مرجع Tidy3D
T3,ph3 = simulate(P3, RES3D)
m3 = metrics(T3); print({k:round(float(v),3) for k,v in m3.items()})
# مقایسه 2D و 3D
T2,_ = simulate(dict(base22, L=56.5), RES2D)
plt.figure(figsize=(6,4)); plt.plot(LAMS,T3[0],label='3D bar'); plt.plot(LAMS,T3[1],label='3D cross')
plt.plot(LAMS,T2[0],'--',label='2D bar'); plt.plot(LAMS,T2[1],'--',label='2D cross'); plt.legend(); plt.grid(alpha=.3)
plt.xlabel('λ (µm)'); save_fig('compare_2D_3D'); plt.show()
# فاز بین دو خروجی در 2x2 باید ≈ 90° باشد (هیبرید)
i0=np.argmin(abs(LAMS-LAM0)); print('Δφ (deg):', np.degrees(((ph3[0][i0]-ph3[1][i0])+np.pi)%(2*np.pi)-np.pi))

In [ ]:
# سوییپ 3D طول حول بهینه (چند نقطه)  
df_L3 = sweep('sweepL_2x2_3D', P3, 'L', [52.0,54.0,56.5,59.0,61.0], res=RES3D)
df_L3[['L','T0','T1','IL_dB','imbalance_dB','BW1dB_both_nm']]

## ۱۲) جدول نهایی (Deliverable) + ذخیره در Drive
ستون‌ها: Insertion loss، Split ratio، Imbalance، پهنای باند ۱dB (طبق تعریف بخش ۱).

In [ ]:
final=[]
def add(name, P, res):
    T,_=simulate(P,res); m=metrics(T); m['Design']=name; m['model']=f"{P['dim']}D"; final.append(m)
add('MMI 2x2 W=4 (opt L)',  dict(base22,L=L22), RES2D)
add('MMI 2x2 W=4 L=56.5 (3D)', P3, RES3D)
add('MMI 1x2 W=6 (opt L)',  dict(base12,L=L12), RES2D)
add('Y-splitter (Zhang 2013)+Sbend', bY, RES2D)
tab = pd.DataFrame(final)[['Design','model','IL_dB','EL_dB','imbalance_dB','split_ratio','BW1dB_IL_nm','BW1dB_imb_nm','BW1dB_both_nm']].round(3)
tab.to_csv(f'{ROOT}/tables/FINAL_table.csv', index=False)
with open(f'{ROOT}/tables/FINAL_table.md','w') as f: f.write(tab.to_markdown(index=False))
tab

## ۱۳) بعداً بدون شبیه‌سازی دوباره
فقط سلول‌های ۱ تا ۴ (import/Drive/توابع) را اجرا کنید؛ چون `simulate` اول کش را می‌خواند، همهٔ نمودارها و جدول‌ها از `MyDrive/MMI_Project/{cache,figs,tables}` فوری برمی‌گردند.

## مراجع
1. L. B. Soldano, E. C. M. Pennings, JLT 13, 615 (1995) — تئوری self-imaging.
2. Y. Zhang et al., Opt. Express 21, 1310 (2013) — Y-junction.
3. PhotonForge: MMI_Coupler_EME و Y_Splitter (نمونه‌های پیوست).
4. Oskooi et al., Meep, Comput. Phys. Commun. 181, 687 (2010).